In [1]:
from pyrootmemo import Parameter
from pyrootmemo.materials import MultipleRoots, Interface
from pyrootmemo.models import AxialPullout

# Axial pullout

Predict pull-out forces for bundles of roots. This class follows models 
developed by Waldron (1977), Waldron & Dakessian (1981) and the DRAM model
developed by Meijer et al. (2022), in the sense that displacements and 
forces are mobilised due to the interaction between root-soil interface 
resistance and root stiffness.

This model class can take a wide variety of different root behaviours, 
and therefore incorporates the models by a wide variety of authors. for 
example, this class can take into account:

* root breakage
* root slippage
* elastic or elasto-plastic root behaviour
* 'embedded' behaviour (e.g. roots remain fully surrounded by soil) or 
    'surface' behaviour, in which the root length in contact with the soil
    gradually reduces as more and more root length will gradually emerge above
    the soil surface
* root survival functions, which determined whether roots break 'suddenly'
    or gradually as an 'average' root, as govered by the weibull shape 
    parameter. This is implemented by looking at the ratio between the force
    at failure and the currently mobilised force in a root, assuming no 
    breakage.

These models underpin models for roots loaded in direct shear, such as `Waldron`
or `Dram`.


## Create a model object

Define some roots, using the `MultipleRoots` class. The AxialPullout model 
requires the following root attributes to be defined:

* Root diameter: (`diameter`)
* Root tensile strength (`tensile strength`)
* Root elastic stiffness (`elastic_modulus`)
* Root length (`length`)

In addition, in the case of elasto-plastic behaviour it also requires:

* Root yield strength (`yield_strength`)
* Root plastic stiffness (`plastic_modulus`)

In the case of modelling roots being pulled out of a soil surface, the length
of root initially sticking out of the soil surface must be defined:

* Initial root legnth above surface (`length_surface`)

In [2]:
roots = MultipleRoots(
    species = 'test_species',
    diameter = Parameter([1, 2, 3, 4], 'mm'),
    tensile_strength = Parameter([500, 300, 100, 50], 'MPa'),
    elastic_modulus = Parameter([1, 1, 1, 1], 'GPa'),
    length = Parameter([1, 1, 1, 1], 'm'),
    yield_strength = Parameter([200, 150, 50, 25], 'MPa'),
    plastic_modulus = Parameter([0.2, 0.2, 0.2, 0.2], 'GPa'),
    length_surface = Parameter([0, 0, 0, 0], 'm')
    )

We also need to define the root-soil interface behaviour, using the `Interface` class:

In [3]:
interface = Interface(
    shear_strength = Parameter(10, 'kPa')
    )

Next, create an instance of the Axial pullout class. The followin example models:

* Roots being pulled out of the soil surface (`surface = True`)
* Roots tensile failure is allowed (`breakage = True`)
* Root slipping is allowed (`slipping = True`)
* Elasto-plastic root behaviour is enabled (`elastoplastic = True`)

Weibull failure distributions, such as in the RBMw, can be implemented by
specifying the `weibull_shape` input argument. If not defined, roots are assumed
to break suddenly.

In [4]:
pullout = AxialPullout(
    roots, 
    interface,
    surface = True,
    breakage = True,
    slipping = True,
    elastoplastic = True,
    weibull_shape = 5.0
    )

c:\Users\gjm36\AppData\Local\miniforge3\envs\rrmm\Lib\site-packages\pint\facets\numpy\numpy_func.py:322: RuntimeWarning: invalid value encountered in sqrt
  result_magnitude = func(*stripped_args, **stripped_kwargs)


## Calculations

Various calculations can be made. The method calculated results are returned
depends on the "results" input argument that can be defined for each calculation
function:

* `results = "attribute"` (default behaviour): calculated model results are 
stored in a dictionary in the `output` attribute of the class.
* `results = "return"`: functions returns a dictionary with results, without 
writing to class' `output` attribute
* `results = "both"`: functions both returns dictionary output *and* writes 
to the `output` attribute.
 
### Force (`force`)

To calculate the current tensile force in each root at a specific level of 
pullout displacement, first do the calculation and then request the results 
from the `output` attribute:

In [5]:
displacement = Parameter(2.0, 'mm')
pullout.calc_force(displacement)
print(pullout.output['force_per_root'])

[9.87273014865426 27.974979973185004 51.434771733854056 79.22569774432024] newton


In [6]:
print(pullout.output)

{'behaviour_types': array(['Not in tension', 'Anchored, elastic', 'Slipping, elastic',
       'Full pullout', 'Anchored, plastic', 'Slipping, plastic',
       'Slipping, plastic', 'Full pullout'], dtype='<U17'), 'force_per_root': <Quantity([ 9.87273015 27.97497997 51.43477173 79.22569774], 'newton')>, 'behaviour_index': array([1, 1, 1, 1]), 'survival_fraction': array([0.99999999, 0.99999998, 0.99999867, 0.9999792 ])}


Also calculated are:

* The fraction of roots still intact (`survival_fraction`)
* The current behaviour of the root (`behaviour_index`). The name of the 
  behaviours is stored in the `behaviour_types` key of the `output` dictionary

In [7]:
print(pullout.output['survival_fraction'])
print(pullout.output['behaviour_types'][pullout.output['behaviour_index']])

[0.99999999 0.99999998 0.99999867 0.9999792 ]
['Anchored, elastic' 'Anchored, elastic' 'Anchored, elastic'
 'Anchored, elastic']


In this case, because the displacement is very small, roots are mostly still instact and the behaviour is still elastic and anchored (i.e. no slippage).

This may change when the pullout displacement is much larger, e.g.:

In [8]:
displacement = Parameter(100.0, 'mm')
pullout.calc_force(displacement)
print(pullout.output['force_per_root'])
print(pullout.output['survival_fraction'])
print(pullout.output['behaviour_types'][pullout.output['behaviour_index']])

[28.383854435424695 56.6599338092742 84.93264387975164 113.18647612386384] newton
[0.99999806 0.99999918 0.99997339 0.99979632]
['Slipping, elastic' 'Slipping, elastic' 'Slipping, elastic'
 'Slipping, elastic']
